# Recommandation enrichie par LLM (Mistral)

On reprend exactement les candidats produits par `/recommendation` (moteur de similarité déjà construit), et on demande à un LLM de rédiger une recommandation argumentée, sur le ton d'une critique de cinéma. Le LLM ne choisit rien : il ne fait que commenter les films déjà sélectionnés par le moteur de similarité — pas de nouvelle recherche de sa part.

Pré-requis : `MISTRAL_API_KEY` dans `.env`, `data/models/recommandation.joblib` déjà généré (`python src/precompute_recommandation.py`).

## 1. Configuration

In [1]:
import sys
import os

import yaml
from dotenv import load_dotenv

load_dotenv()

with open("../config.yaml", "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
MODELE = CONFIG["llm"]["modele"]
TEMPERATURE = CONFIG["llm"]["temperature"]
MAX_TOKENS = CONFIG["llm"]["max_tokens"]

assert MISTRAL_API_KEY, "Clé API manquante : vérifiez MISTRAL_API_KEY dans votre .env"
print(f"Modèle configuré : {MODELE}")

Modèle configuré : mistral-small-latest


## 2. Réutiliser le moteur de recommandation déjà construit

Pas de nouveau calcul de similarité ici : on importe directement `recommander()` depuis `api/moteur_recommandation.py`, qui charge l'artefact précalculé.

In [2]:
sys.path.append("../api")
from moteur_recommandation import recommander, METADATA

film_exemple = METADATA[METADATA["title"].str.contains("Spider-Man", case=False, na=False)].iloc[0]
print(f"Film de référence : {film_exemple['title']} (id {film_exemple['id']})")

candidats = recommander(film_exemple["id"])
candidats

Film de référence : Spider-Man (id 557)


[{'id': 558,
  'title': 'Spider-Man 2',
  'genres': ['Science Fiction', 'Action', 'Adventure'],
  'runtime': np.int64(127),
  'release_date': '2004-06-25',
  'actors': ['Kirsten Dunst',
   'Alfred Molina',
   'Tobey Maguire',
   'James Franco',
   'Rosemary Harris'],
  'directors': ['Sam Raimi'],
  'score_similarite': 0.406},
 {'id': 559,
  'title': 'Spider-Man 3',
  'genres': ['Science Fiction', 'Action', 'Adventure'],
  'runtime': np.int64(139),
  'release_date': '2007-05-01',
  'actors': ['Kirsten Dunst',
   'Tobey Maguire',
   'James Franco',
   'Topher Grace',
   'Thomas Haden Church'],
  'directors': ['Sam Raimi'],
  'score_similarite': 0.399},
 {'id': 155,
  'title': 'The Dark Knight',
  'genres': ['Action', 'Thriller', 'Crime'],
  'runtime': np.int64(152),
  'release_date': '2008-07-16',
  'actors': ['Maggie Gyllenhaal',
   'Heath Ledger',
   'Christian Bale',
   'Michael Caine',
   'Aaron Eckhart'],
  'directors': ['Christopher Nolan'],
  'score_similarite': 0.181},
 {'id': 38

## 3. Construire le prompt

Consigne explicite de ne pas inventer d'information au-delà de ce qui est fourni — important pour un usage "grounded" sur des données réelles plutôt qu'une réponse en roue libre.

In [3]:
def construire_prompt(titre_source, candidats):
    lignes = []
    for film in candidats:
        genres = ", ".join(film["genres"]) if film["genres"] else "genres non renseignés"
        acteurs = ", ".join(film["actors"][:3]) if film["actors"] else "casting non renseigné"
        realisateurs = ", ".join(film["directors"]) if film["directors"] else "réalisateur non renseigné"
        lignes.append(
            f"- {film['title']} ({genres}) — réalisé par {realisateurs}, "
            f"avec {acteurs}, score de similarité {film['score_similarite']}"
        )
    liste_candidats = "\n".join(lignes)

    return (
        f"Un utilisateur vient d'apprécier le film \"{titre_source}\".\n"
        f"Voici {len(candidats)} films sélectionnés algorithmiquement pour leur proximité "
        f"de genre, casting et thématique :\n\n{liste_candidats}\n\n"
        f"Rédige une recommandation courte (150 mots maximum), sur le ton d'une critique "
        f"de cinéma argumentée : explique pourquoi ces films devraient plaire à quelqu'un "
        f"qui a aimé \"{titre_source}\", en t'appuyant uniquement sur les informations "
        f"ci-dessus (genre, réalisateur, acteurs). N'invente aucune information absente "
        f"de la liste."
    )


prompt = construire_prompt(film_exemple["title"], candidats)
print(prompt)

Un utilisateur vient d'apprécier le film "Spider-Man".
Voici 5 films sélectionnés algorithmiquement pour leur proximité de genre, casting et thématique :

- Spider-Man 2 (Science Fiction, Action, Adventure) — réalisé par Sam Raimi, avec Kirsten Dunst, Alfred Molina, Tobey Maguire, score de similarité 0.406
- Spider-Man 3 (Science Fiction, Action, Adventure) — réalisé par Sam Raimi, avec Kirsten Dunst, Tobey Maguire, James Franco, score de similarité 0.399
- The Dark Knight (Action, Thriller, Crime) — réalisé par Christopher Nolan, avec Maggie Gyllenhaal, Heath Ledger, Christian Bale, score de similarité 0.181
- Megamind (Science Fiction, Action, Comedy, Family, Animation) — réalisé par réalisateur non renseigné, avec David Cross, Brad Pitt, Tom McGrath, score de similarité 0.17
- Man of Steel (Science Fiction, Action, Adventure) — réalisé par Zack Snyder, avec Michael Shannon, Russell Crowe, Diane Lane, score de similarité 0.165

Rédige une recommandation courte (150 mots maximum), sur

## 4. Appeler Mistral

In [4]:
# Identifier les modèles disponibles dans l'API Mistral
from mistralai import Mistral

client = Mistral(api_key=MISTRAL_API_KEY)

for modele in client.models.list().data:
    print(modele.id)

ImportError: cannot import name 'Mistral' from 'mistralai' (unknown location)

In [6]:
import sys; print(sys.executable)

c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2026_CINE_DATA\CineData\venv\Scripts\python.exe


In [5]:
import mistralai
print(mistralai.__file__)

None


In [5]:
from mistralai import Mistral

client = Mistral(api_key=MISTRAL_API_KEY)


def generer_argumentaire(titre_source, candidats):
    """
    Appelle Mistral pour rédiger la recommandation argumentée.
    Retourne le texte, ou None en cas d'erreur (timeout, quota, panne...).
    """
    prompt = construire_prompt(titre_source, candidats)

    try:
        reponse = client.chat.complete(
            model=MODELE,
            messages=[
                {
                    "role": "system",
                    "content": "Tu es un critique de cinéma qui recommande des films de façon argumentée et concise.",
                },
                {"role": "user", "content": prompt},
            ],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
        )
        return reponse.choices[0].message.content
    except Exception as erreur:
        print(f"Erreur lors de l'appel à Mistral : {erreur}")
        return None

ImportError: cannot import name 'Mistral' from 'mistralai' (unknown location)

In [ ]:
argumentaire = generer_argumentaire(film_exemple["title"], candidats)
print(argumentaire)

## 5. Résultat combiné

La forme retenue pour l'API : le texte argumenté **et** la liste structurée (comme `/recommendation`) — utile pour vérifier/comparer, et transparent sur ce qui a été donné au LLM comme contexte.

In [ ]:
resultat = {
    "film_source": film_exemple["title"],
    "argumentaire": argumentaire,
    "films_recommandes": candidats,
}

resultat

---
**Étape suivante (pas faite ici) :** transposer `construire_prompt()` et `generer_argumentaire()` dans un module `api/assistant_llm.py`, puis une route `GET /enhanced_recommendation/{film_id}` qui réutilise `recommander()` + ce module, avec un `HTTPException(503, ...)` explicite si l'appel à Mistral échoue.